In [1]:
# !pip install langchain
# !pip install langchain-community
# !pip install -U langchain-google-genai
# !pip install --upgrade langchain-core langchain-google-vertexai
# !pip install --upgrade langchain-community duckduckgo-search
# !pip install langgraph
# !pip install requests pubchempy rdkit-pypi
# !pip install ctx-python
# !pip install biopython
# !pip install langgraph-checkpoint-sqlite
# !pip uninstall -y faiss-cpu
# !pip install langchain_openai python-dotenv

In [2]:
# export LANGSMITH_TRACING="true"
# export LANGSMITH_API_KEY="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"
# export LANGSMITH_PROJECT="default" # or any other project name
# openai_api_key = "REMOVED_CREDENTIAL"

In [1]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr

load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)


# model = ChatOpenRouter(model_name="google/gemini-2.0-flash-001")

In [ ]:
import datetime
import logging
import re, logging, warnings, sqlite3, os, time, random, pickle, requests
from pathlib import Path
from typing import List, Dict
from xml.etree import ElementTree as ET
from http.client import IncompleteRead

from langchain.memory import (
    ConversationBufferMemory,
    ConversationEntityMemory,
    CombinedMemory,
)
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chains import RetrievalQA
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt
from typing import Dict, List, Any, Optional, Type
from dataclasses import dataclass

from pydantic import PrivateAttr, BaseModel, Field
from pydantic import ValidationError
from langchain_core.tools import BaseTool

# Import MemorySaver for short-term memory persistence
from langgraph.checkpoint.memory import MemorySaver

from langchain.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper

import logging


logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("log/log")


import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Descriptors
RDKIT_AVAILABLE = True


# PubMed Entrez for journal search
from Bio import Entrez
Entrez.email = "laozc@student.ubc.ca"



def safe_tool_wrapper(func):
    """Wrapper to ensure tools always return non-empty strings and catch exceptions,
    reporting which class and function raised the error."""
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            if not result or (isinstance(result, str) and result.strip() == ""):
                return "No information available for this query."
            return str(result).strip()
        except Exception as e:
            # Determine class name if this is a bound method
            if hasattr(func, "__self__") and func.__self__:
                cls_name = func.__self__.__class__.__name__
            else:
                # Fallback: parse the qualname
                cls_name = func.__qualname__.split(".")[0]
            func_name = func.__name__
            log.error(f"Tool '{cls_name}.{func_name}' execution error: {e}")
            return f"Tool '{cls_name}.{func_name}' error occurred: {str(e)}. Please try a different approach."
    return wrapper

class CompoundDatabaseInput(BaseModel):
    compound_name: Optional[str] = Field(None, description="Name of the compound")
    mass: Optional[float] = Field(None, description="Molecular mass")
    formula: Optional[str] = Field(None, description="Molecular formula")
    smiles: Optional[str] = Field(None, description="SMILES notation")
    
class CompoundDatabaseTool(BaseTool):
    name: str = "compound_database"
    description: str = "Look up compound information from PubChem, HMDB, and other databases"
    args_schema: Type[BaseModel] = CompoundDatabaseInput


    @safe_tool_wrapper
    def _run(self,
            compound_name: Optional[str] = None,
            mass: Optional[float] = None,
            formula: Optional[str] = None,
            smiles: Optional[str] = None) -> str:

        if not any([compound_name, mass, formula, smiles]):
            return "Please provide at least one of: compound_name, mass, formula, or SMILES."

        if not RDKIT_AVAILABLE:
            return "RDKit/PubChemPy not installed. Run pip install rdkit-pypi pubchempy."

        results = []

        # Name-based search
        if compound_name:
            try:
                results.extend(self._search_by_name(compound_name))
            except Exception as e:
                log.warning(f"Name search failed: {e}")
        
        # Mass-based search
        if mass:
            try:
                compounds = pcp.get_compounds(str(mass), 'mass', listkey_count=5)
                for cmpd in compounds[:3]:
                    smiles = getattr(cmpd, 'connectivity_smiles', None) or getattr(cmpd, 'canonical_smiles', None)
                    if smiles:
                        mol = Chem.MolFromSmiles(smiles)
                        mw = Descriptors.MolWt(mol) if mol else None
                        results.append({
                            'name': cmpd.iupac_name or (cmpd.synonyms[0] if cmpd.synonyms else 'Unknown'),
                            'formula': cmpd.molecular_formula,
                            'mass': round(mw, 4) if mw else mass,
                            'smiles': smiles,
                            'cid': cmpd.cid
                        })
            except Exception as e:
                log.warning(f"Mass search failed: {e}")

        # Formula-based search
        if formula:
            try:
                results.extend(self._search_by_formula(formula))
            except Exception as e:
                log.warning(f"Formula search failed: {e}")
        
        # SMILES-based search - FIXED: Search PubChem first, then analyze
        if smiles:
            try:
                # First, search PubChem for this SMILES
                smiles_results = self._search_by_smiles(smiles)
                results.extend(smiles_results)
            except Exception as e:
                log.warning(f"SMILES search failed: {e}")

        # Format results if we found any compounds
        if results:
            # Remove duplicates based on CID
            unique_results = []
            seen_cids = set()
            for r in results:
                cid = r.get('cid')
                if cid and cid not in seen_cids:
                    unique_results.append(r)
                    seen_cids.add(cid)
                elif not cid:  # Include results without CID
                    unique_results.append(r)
            
            
            # Format as table
            table = "## Compound Search Results\n\n"
            table += "| Name | Formula | Mass (Da) | SMILES | PubChem CID |\n"
            table += "|------|---------|-----------|--------|-------------|\n"

            for r in unique_results[:5]:  # Limit to top 5 results
                name = r.get('name', 'N/A')
                # Don't truncate names too aggressively
                if name and len(name) > 40:
                    name = name[:37] + "..."
                
                smiles_display = r.get('smiles', 'N/A')
                if smiles_display and len(smiles_display) > 25:
                    smiles_display = smiles_display[:22] + "..."
                
                cid = r.get('cid', 'N/A')
                pubchem_link = f"[{cid}](https://pubchem.ncbi.nlm.nih.gov/compound/{cid})" if cid != 'N/A' else 'N/A'
                
                table += (
                    f"| {name or 'N/A'} | {r.get('formula', 'N/A')} | {r.get('mass', 'N/A')} | "
                    f"{smiles_display or 'N/A'} | {pubchem_link} |\n"
                )

            # Add detailed information for the first result if available
            if unique_results:
                first_result = unique_results[0]
                table += f"\n### Primary Result Details:\n"
                table += f"**Name:** {first_result.get('name', 'N/A')}\n"
                table += f"**Formula:** {first_result.get('formula', 'N/A')}\n"
                table += f"**Molecular Weight:** {first_result.get('mass', 'N/A')} Da\n"
                table += f"**PubChem CID:** {first_result.get('cid', 'N/A')}\n"
                table += f"**SMILES:** {first_result.get('smiles', 'N/A')}\n"
                
                # Add synonyms if available
                if 'synonyms' in first_result and first_result['synonyms']:
                    table += f"**Common Names:** {', '.join(first_result['synonyms'][:3])}\n"

            # Add molecular analysis if SMILES was provided
            if smiles:
                analysis = self._analyze_smiles_properties(smiles)
                table += f"\n### Molecular Analysis:\n{analysis}"

            return table

        # If no results from PubChem searches, provide molecular analysis for SMILES
        if smiles:
            return self._analyze_smiles(smiles)
        
        # If no results from any search method
        if mass:
            return self._search_by_mass(mass)
        
        return "No compounds found. Try a different search term or check the spelling."
  

    def _search_by_smiles(self, smiles: str) -> List[Dict]:
        """Search compounds by SMILES string"""
        try:
            # Search PubChem by SMILES
            compounds = pcp.get_compounds(smiles, 'smiles', listkey_count=5)
            results = []
            
            for compound in compounds[:3]:
                try:
                    compound_smiles = getattr(compound, 'connectivity_smiles', None) or getattr(compound, 'canonical_smiles', None)
                    if compound_smiles:
                        mol = Chem.MolFromSmiles(compound_smiles)
                        if mol:
                            mw = Descriptors.MolWt(mol)
                            
                            # IMPROVED name selection logic
                            name = None
                            
                            # First priority: Look for common vitamin/supplement names
                            if compound.synonyms:
                                # Second priority: Short, simple names (likely common names)
                                if not name:
                                    for synonym in compound.synonyms:
                                        if len(synonym) < 15 and not any(char in synonym for char in ['-', '(', ')', '[', ']', '{']):
                                            name = synonym
                                            break
                            
                            # Third priority: IUPAC name
                            if not name:
                                name = compound.iupac_name
                            
                            # Fourth priority: First synonym
                            if not name and compound.synonyms:
                                name = compound.synonyms[0]
                            
                            # Last resort: CID
                            if not name:
                                name = f"CID {compound.cid}"
                            
                            results.append({
                                'name': name,
                                'formula': compound.molecular_formula,
                                'mass': round(mw, 4),
                                'smiles': compound_smiles,
                                'cid': compound.cid,
                                'synonyms': compound.synonyms[:5] if compound.synonyms else []
                            })
                except Exception as e:
                    log.warning(f"Error processing compound {getattr(compound, 'cid', 'unknown')}: {e}")
                    continue
                    
            return results
        except Exception as e:
            log.warning(f"SMILES search failed: {e}")
            return []
        
    def _analyze_smiles_properties(self, smiles: str) -> str:
        """Analyze SMILES string and return molecular properties (shorter version for tables)"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if not mol:
                return f"Invalid SMILES: {smiles}"
            
            mw = Descriptors.MolWt(mol)
            formula = Chem.rdMolDescriptors.CalcMolFormula(mol)
            exact_mass = Descriptors.ExactMolWt(mol)
            
            return (
                f"- Formula: {formula}\n",
                f"- Molecular Weight: {mw:.4f} Da\n" ,
                f"- Exact Mass: {exact_mass:.4f} Da"
            )
        except Exception as e:
            return f"Error analyzing SMILES: {str(e)}"

    def _search_by_name(self, compound_name: str) -> List[Dict]:
        """Search compounds by name with improved error handling"""
        try:
            compounds = pcp.get_compounds(compound_name, 'name', listkey_count=5)
        except Exception as e:
            log.warning(f"PubChem name search failed for '{compound_name}': {e}")
            return []
            
        results = []
        
        for compound in compounds[:3]:
            try:
                smiles = getattr(compound, 'connectivity_smiles', None) or getattr(compound, 'canonical_smiles', None)
                if smiles:
                    mol = Chem.MolFromSmiles(smiles)
                    if mol:
                        mw = Descriptors.MolWt(mol)
                        # Better name selection
                        name = compound.iupac_name
                        if not name and compound.synonyms:
                            name = compound.synonyms[0]
                        if not name:
                            name = compound_name
                        
                        results.append({
                            'name': name,
                            'formula': compound.molecular_formula,
                            'mass': round(mw, 4),
                            'smiles': smiles,
                            'cid': compound.cid
                        })
            except Exception as e:
                log.warning(f"Error processing compound {getattr(compound, 'cid', 'unknown')}: {e}")
                continue
                
        return results

    def _search_by_formula(self, formula: str) -> List[Dict]:
        """Search compounds by molecular formula"""
        try:
            compounds = pcp.get_compounds(formula, 'formula', listkey_count=5)
        except Exception as e:
            log.warning(f"PubChem formula search failed for '{formula}': {e}")
            return []
            
        results = []
        
        for compound in compounds[:3]:
            try:
                smiles = getattr(compound, 'connectivity_smiles', None) or getattr(compound, 'canonical_smiles', None)
                # Better name selection
                name = compound.iupac_name
                if not name and compound.synonyms:
                    name = compound.synonyms[0]
                if not name:
                    name = f"CID {compound.cid}"
                    
                # Calculate mass if we have smiles
                mass = None
                if smiles:
                    try:
                        mol = Chem.MolFromSmiles(smiles)
                        if mol:
                            mass = round(Descriptors.MolWt(mol), 4)
                    except:
                        pass
                        
                results.append({
                    'name': name,
                    'formula': compound.molecular_formula,
                    'smiles': smiles,
                    'mass': mass,
                    'cid': compound.cid
                })
            except Exception as e:
                log.warning(f"Error processing compound {getattr(compound, 'cid', 'unknown')}: {e}")
                continue
                
        return results

    def _analyze_smiles(self, smiles: str) -> str:
        """Analyze SMILES string and return molecular properties"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if not mol:
                return f"Invalid SMILES string: {smiles}. Please check the format."
            
            mw = Descriptors.MolWt(mol)
            formula = Chem.rdMolDescriptors.CalcMolFormula(mol)
            
            return (
                f"SMILES Analysis Results:\n",
                f"SMILES: {smiles}\n",
                f"Molecular Formula: {formula}\n",
                f"Molecular Weight: {mw:.4f} Da\n",
                f"Exact Mass: {Descriptors.ExactMolWt(mol):.4f} Da"
            )
        except Exception as e:
            return f"Error analyzing SMILES: {str(e)}"

    def _search_by_mass(self, mass: float) -> str:
        """Provide guidance for mass-based searches"""
        return (
            f"Mass-based search for m/z {mass}:\n\n",
            "Direct mass search requires specialized database APIs. Consider these approaches:\n\n",
            "1. **Manual Database Search:**\n",
            "   - PubChem: Use Advanced Search → Molecular Weight filter\n",
            "   - HMDB: Use MS Search tool\n",
            "   - METLIN: Personal Metabolite Database\n",
            "   - ChemSpider: Structure Search by molecular weight\n\n",
            "2. **Mass Tolerance Considerations:**\n",
            f"   - Low resolution: ±0.5 Da ({mass-0.5:.1f} - {mass+0.5:.1f})\n",
            f"   - High resolution: ±5 ppm ({mass*(1-5e-6):.4f} - {mass*(1+5e-6):.4f})\n\n",
            "3. **Adduct Considerations:**\n",
            f"   - [M+H]+: {mass-1.007:.3f} Da\n",
            f"   - [M+Na]+: {mass-22.990:.3f} Da\n",
            f"   - [M-H]-: {mass+1.007:.3f} Da"
        )

    def _format_results(self, results: List[Dict]) -> str:
        """Format search results for display"""
        if not results:
            return "No results found."
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"**Result {i}:**\n",
                f"Name: {result.get('name', 'N/A')}\n",
                f"Formula: {result.get('formula', 'N/A')}\n",
                f"Mass: {result.get('mass', 'N/A')} Da\n",
                f"SMILES: {result.get('smiles', 'N/A')}\n",
                f"PubChem CID: {result.get('cid', 'N/A')}"
            )
        
        return "\n\n".join(formatted_results)


class FullTextJournalSearchInput(BaseModel):
    query: str = Field(..., description="Academic search terms")
    max_results: int = Field(20, description="Maximum number of results to return")
    include_preprints: bool = Field(True, description="Include preprint servers")

class FullTextJournalSearchTool(BaseTool):
    name: str = "fulltext_journal_search"
    description: str = "Fetch peer-reviewed articles from multiple databases including PubMed/PMC, arXiv, bioRxiv, and CrossRef."
    args_schema: Type[FullTextJournalSearchInput] = FullTextJournalSearchInput

    _entrez_email: str = PrivateAttr("laozc@student.ubc.ca")
    _max_retries:   int = PrivateAttr(3)
    _backoff_base:  int = PrivateAttr(2)

    def __init__(self, **data):
        super().__init__(**data)
        
    def entrez_call(self, func, *args, **kwargs):
        """Helper to perform Entrez calls with retries"""
        for attempt in range(self._max_retries):
            try:
                return func(*args, **kwargs)
            except (IncompleteRead, Exception) as e:
                log.warning(f"Entrez call error (attempt {attempt+1}): {e}")
                if attempt < self._max_retries - 1:
                    delay = self._backoff_base ** attempt
                    time.sleep(delay)
                else:
                    raise

    def search_pubmed_pmc(self, query: str, max_results: int) -> List[Dict]:
        """Search PubMed/PMC for articles"""
        Entrez.email = self._entrez_email
        results = []
        
        try:
            # Search PubMed
            handle = self.entrez_call(Entrez.esearch, db="pubmed", term=query, 
                                    retmax=max_results, sort="relevance")
            search_results = self.entrez_call(Entrez.read, handle)
            pmids = search_results.get("IdList", [])
            
            if not pmids:
                return results
                
            # Get summaries
            summary_handle = self.entrez_call(
                Entrez.esummary, db="pubmed", id=",".join(pmids), retmode="xml"
            )
            records = self.entrez_call(Entrez.read, summary_handle)
            
            for rec in records:
                pmid = rec["Id"]
                authors = rec.get("AuthorList", [])
                author_str = ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else "")
                
                article_data = {
                    "pmid": pmid,
                    "title": rec.get("Title", ""),
                    "authors": author_str,
                    "journal": rec.get("FullJournalName") or rec.get("Source", ""),
                    "pubdate": rec.get("PubDate", ""),
                    "volume": rec.get("Volume", ""),
                    "issue": rec.get("Issue", ""),
                    "pages": rec.get("Pages", ""),
                    "doi": rec.get("DOI", ""),
                    "source": "PubMed"
                }
                
                # Get abstract and full text (methods remain available separately)
                try:
                    art_handle = self.entrez_call(Entrez.efetch, db="pubmed", id=pmid, retmode="xml")
                    art = self.entrez_call(Entrez.read, art_handle)
                    abstract = " ".join(
                        art["PubmedArticle"][0]["MedlineCitation"]["Article"]
                           .get("Abstract", {}).get("AbstractText", [])
                    ) or "(No abstract available.)"
                    article_data["abstract"] = abstract
                    
                    # Try to get full text from PMC (now returns both full text and methods)
                    full_text, methods_text = self.get_pmc_fulltext(pmid)
                    article_data["full_text"] = full_text
                    article_data["methods"] = methods_text
                    article_data["has_fulltext"] = bool(full_text or methods_text)
                    
                except Exception as e:
                    log.error(f"Error fetching details for PMID {pmid}: {e}")
                    article_data["abstract"] = "(Abstract fetch failed.)"
                    article_data["methods"] = ""
                    article_data["full_text"] = ""
                    article_data["has_fulltext"] = False
                
                results.append(article_data)
                
        except Exception as e:
            log.error(f"PubMed search error: {e}")
            
        return results

    def get_pmc_fulltext(self, pmid: str) -> tuple[str, str]:
        """Fetch PMC full text for a PubMed ID if available.

        Returns a tuple (full_text, methods_text). Both are empty strings if not found.
        """
        try:
            link_handle = self.entrez_call(Entrez.elink, dbfrom="pubmed", db="pmc", id=pmid)
            link_data = self.entrez_call(Entrez.read, link_handle)
            pmcids = []
            for ls in link_data:
                for lset in ls.get("LinkSetDb", []):
                    if lset.get("DbTo") == "pmc":
                        for lnk in lset.get("Link", []):
                            if "Id" in lnk:
                                pmcids.append(lnk["Id"])

            if pmcids:
                pmc_handle = self.entrez_call(Entrez.efetch, db="pmc", id=pmcids[0], retmode="xml")
                root = ET.parse(pmc_handle).getroot()

                # Build full text by concatenating sections under <body>
                full_text_parts = []
                methods_parts = []

                # Look for body sections first
                for body in root.findall('.//body'):
                    for sec in body.findall('.//sec'):
                        title = sec.findtext('title') or ""
                        text = ''.join(sec.itertext()).strip()
                        if title:
                            full_text_parts.append(f"## {title}\n{text}")
                        else:
                            full_text_parts.append(text)

                        # If section looks like methods, also capture separately
                        if any(keyword in title.lower() for keyword in ['method', 'procedure', 'protocol', 'experiment', 'materials']):
                            methods_parts.append(f"## {title}\n{text}")

                # Fallback: if no <body> found, gather top-level sec nodes
                if not full_text_parts:
                    for sec in root.findall('.//sec'):
                        title = sec.findtext('title') or ""
                        text = ''.join(sec.itertext()).strip()
                        if title:
                            full_text_parts.append(f"## {title}\n{text}")
                        else:
                            full_text_parts.append(text)
                        if any(keyword in title.lower() for keyword in ['method', 'procedure', 'protocol', 'experiment', 'materials']):
                            methods_parts.append(f"## {title}\n{text}")

                full_text = "\n\n".join(full_text_parts).strip()
                methods_text = "\n\n".join(methods_parts).strip()
                return full_text, methods_text

        except Exception as e:
            log.error(f"PMC full-text extraction error for PMID {pmid}: {e}")

        return "", ""

    def search_alternative_sources(self, article_info: Dict, max_per_source: int = 3) -> List[Dict]:
        """Search for a specific article in alternative databases using title, authors, DOI"""
        results = []
        title = article_info.get('title', '')
        authors = article_info.get('authors', '')
        doi = article_info.get('doi', '')
        
        # Create search queries for different databases
        search_queries = []
        
        # Primary search by title (most reliable)
        if title:
            # Clean title for search
            clean_title = ' '.join(title.split()[:8])  # First 8 words usually sufficient
            search_queries.append(clean_title)
        
        # Secondary search by DOI if available
        if doi:
            search_queries.append(doi)
        
        # Search each alternative database
        for query in search_queries:
            if len(results) >= max_per_source:
                break
                
            # Search arXiv
            arxiv_results = self.search_arxiv_specific(query, title, authors, max_per_source)
            results.extend(arxiv_results)
            
            # Search bioRxiv
            biorxiv_results = self.search_biorxiv_specific(query, title, authors, max_per_source)
            results.extend(biorxiv_results)
            
            # Search CrossRef
            crossref_results = self.search_crossref_specific(query, title, authors, doi, max_per_source)
            results.extend(crossref_results)
        
        return results

    def search_arxiv_specific(self, query: str, target_title: str, target_authors: str, max_results: int) -> List[Dict]:
        """Search arXiv for a specific article"""
        results = []
        try:
            url = "http://export.arxiv.org/api/query"
            params = {
                'search_query': f'all:"{query}"',
                'start': 0,
                'max_results': max_results * 2,  # Get more to filter better
                'sortBy': 'relevance',
                'sortOrder': 'descending'
            }
            
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            
            root = ET.fromstring(response.content)
            ns = {'atom': 'http://www.w3.org/2005/Atom'}
            
            for entry in root.findall('atom:entry', ns):
                title = entry.find('atom:title', ns)
                title_text = title.text.strip().replace('\n', ' ') if title is not None else ""
                
                # Check if this matches our target article
                if self.is_same_article(title_text, target_title, target_authors):
                    summary = entry.find('atom:summary', ns)
                    abstract = summary.text.strip() if summary is not None else ""
                    
                    authors = []
                    for author in entry.findall('atom:author', ns):
                        name = author.find('atom:name', ns)
                        if name is not None:
                            authors.append(name.text)
                    
                    published = entry.find('atom:published', ns)
                    pub_date = published.text[:10] if published is not None else ""
                    
                    arxiv_id = entry.find('atom:id', ns)
                    arxiv_url = arxiv_id.text if arxiv_id is not None else ""
                    
                    results.append({
                        "title": title_text,
                        "authors": ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else ""),
                        "abstract": abstract,
                        "pubdate": pub_date,
                        "url": arxiv_url,
                        "source": "arXiv",
                        "has_fulltext": True,
                        "methods": "",
                        "match_confidence": "high"
                    })
                    
                    if len(results) >= max_results:
                        break
                        
        except Exception as e:
            log.error(f"arXiv specific search error: {e}")
            
        return results

    def search_biorxiv_specific(self, query: str, target_title: str, target_authors: str, max_results: int) -> List[Dict]:
        """Search bioRxiv for a specific article"""
        results = []
        try:
            from datetime import datetime, timedelta
            end_date = datetime.now()
            start_date = end_date - timedelta(days=1095)  # 3 years back for better coverage
            
            start_str = start_date.strftime("%Y-%m-%d")
            end_str = end_date.strftime("%Y-%m-%d")
            
            url = f"https://api.biorxiv.org/details/biorxiv/{start_str}/{end_str}"
            
            headers = {
                'User-Agent': 'Enhanced-Journal-Search/1.0 (mailto:laozc@student.ubc.ca)'
            }
            
            response = requests.get(url, headers=headers, timeout=60)
            response.raise_for_status()
            
            data = response.json()
            
            if data.get('messages') and data['messages'][0].get('status') == 'ok':
                papers = data.get('collection', [])
                
                for paper in papers:
                    paper_title = paper.get('title', '')
                    paper_authors = paper.get('authors', '')

                    # If a query string was provided, prioritize matching the query in the title
                    if query and query.strip():
                        if query.lower() not in paper_title.lower():
                            continue
                    
                    # Check if this matches our target article
                    if self.is_same_article(paper_title, target_title, target_authors):
                        # Format authors
                        if paper_authors:
                            author_list = []
                            for author in paper_authors.split(';'):
                                author = author.strip()
                                if ',' in author:
                                    last, first = author.split(',', 1)
                                    author_list.append(f"{first.strip()} {last.strip()}")
                                else:
                                    author_list.append(author)
                            
                            formatted_authors = ", ".join(author_list[:3])
                            if len(author_list) > 3:
                                formatted_authors += " et al."
                        else:
                            formatted_authors = "Authors not available"
                        
                        doi = paper.get('doi', '')
                        paper_url = f"https://www.biorxiv.org/content/{doi}" if doi else ""
                        
                        results.append({
                            "title": paper_title,
                            "authors": formatted_authors,
                            "abstract": paper.get('abstract', ''),
                            "pubdate": paper.get('date', ''),
                            "doi": doi,
                            "url": paper_url,
                            "source": "bioRxiv",
                            "journal": "bioRxiv preprint",
                            "has_fulltext": True,
                            "methods": "",
                            "match_confidence": "high"
                        })
                        
                        if len(results) >= max_results:
                            break
                            
        except Exception as e:
            log.error(f"bioRxiv specific search error: {e}")
            
        return results

    def search_crossref_specific(self, query: str, target_title: str, target_authors: str, doi: str, max_results: int) -> List[Dict]:
        """Search CrossRef for a specific article"""
        results = []
        try:
            # If we have a DOI, search by DOI first
            if doi:
                url = f"https://api.crossref.org/works/{doi}"
                headers = {
                    'User-Agent': 'Enhanced-Journal-Search/1.0 (mailto:laozc@student.ubc.ca)'
                }
                
                response = requests.get(url, headers=headers, timeout=30)
                if response.status_code == 200:
                    data = response.json()
                    item = data.get('message', {})
                    
                    if item:
                        authors = []
                        for author in item.get('author', []):
                            given = author.get('given', '')
                            family = author.get('family', '')
                            if given and family:
                                authors.append(f"{given} {family}")
                        
                        title = ' '.join(item.get('title', []))
                        journal = ' '.join(item.get('container-title', []))
                        
                        results.append({
                            "title": title,
                            "authors": ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else ""),
                            "journal": journal,
                            "pubdate": str(item.get('published-print', {}).get('date-parts', [['']])[0][0] or 
                                      item.get('published-online', {}).get('date-parts', [['']])[0][0] or ''),
                            "doi": item.get('DOI', ''),
                            "url": item.get('URL', ''),
                            "source": "CrossRef",
                            "abstract": "(Abstract not available from CrossRef)",
                            "methods": "",
                            "has_fulltext": bool(item.get('URL')),
                            "match_confidence": "high"
                        })
                        return results
            
            # Fallback to title search - prefer the incoming `query` if provided for broader matching
            url = "https://api.crossref.org/works"
            params = {
                'query.title': query or target_title,
                'rows': max_results * 2,
                'sort': 'relevance',
                'order': 'desc'
            }
            
            headers = {
                'User-Agent': 'Enhanced-Journal-Search/1.0 (mailto:laozc@student.ubc.ca)',
            }
            
            response = requests.get(url, params=params, headers=headers, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            
            for item in data.get('message', {}).get('items', []):
                item_title = ' '.join(item.get('title', []))
                
                # Check if this matches our target article
                if self.is_same_article(item_title, target_title, target_authors):
                    authors = []
                    for author in item.get('author', []):
                        given = author.get('given', '')
                        family = author.get('family', '')
                        if given and family:
                            authors.append(f"{given} {family}")
                    
                    journal = ' '.join(item.get('container-title', []))
                    
                    results.append({
                        "title": item_title,
                        "authors": ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else ""),
                        "journal": journal,
                        "pubdate": str(item.get('published-print', {}).get('date-parts', [['']])[0][0] or 
                                  item.get('published-online', {}).get('date-parts', [['']])[0][0] or ''),
                        "doi": item.get('DOI', ''),
                        "url": item.get('URL', ''),
                        "source": "CrossRef",
                        "abstract": "(Abstract not available from CrossRef)",
                        "methods": "",
                        "has_fulltext": bool(item.get('URL')),
                        "match_confidence": "medium"
                    })
                    
                    if len(results) >= max_results:
                        break
                        
        except Exception as e:
            log.error(f"CrossRef specific search error: {e}")
            
        return results

    def is_same_article(self, title1: str, title2: str, authors1: str = "", authors2: str = "") -> bool:
        """Check if two articles are the same based on title similarity and author overlap"""
        if not title1 or not title2:
            return False
        
        # Normalize titles for comparison
        def normalize_title(title):
            import re
            # Remove punctuation and convert to lowercase
            title = re.sub(r'[^\w\s]', '', title.lower())
            # Remove common words that don't help with matching
            stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'must', 'can', 'shall'}
            words = [w for w in title.split() if w not in stop_words and len(w) > 2]
            return ' '.join(words)
        
        norm_title1 = normalize_title(title1)
        norm_title2 = normalize_title(title2)
        
        # Calculate word overlap ratio
        words1 = set(norm_title1.split())
        words2 = set(norm_title2.split())
        
        if not words1 or not words2:
            return False
        
        overlap = len(words1.intersection(words2))
        total_unique = len(words1.union(words2))
        
        # Consider it a match if >70% of words overlap
        similarity_ratio = overlap / total_unique if total_unique > 0 else 0
        
        return similarity_ratio > 0.7

    def format_results(self, all_results: List[Dict]) -> List[str]:
        """Format all results into readable snippets"""
        snippets = []
        
        for result in all_results:
            # Create citation
            citation_parts = []
            if result.get("authors"):
                citation_parts.append(result["authors"])
            if result.get("title"):
                citation_parts.append(result["title"])
            if result.get("journal"):
                citation_parts.append(result["journal"])
            if result.get("pubdate"):
                citation_parts.append(result["pubdate"])
            if result.get("doi"):
                citation_parts.append(f"DOI: {result['doi']}")
            
            citation = ". ".join(filter(None, citation_parts))
            
            # Build snippet
            snippet_parts = [
                f"**Source: {result.get('source', 'Unknown')}**",
                f"**Citation:** {citation}",
            ]
            
            if result.get("abstract"):
                snippet_parts.append(f"**Abstract:** {result['abstract']}")
            
            # Include a truncated full-text excerpt if available
            full_text = result.get("full_text", "") or ""
            if full_text:
                excerpt = (full_text[:1000] + '...') if len(full_text) > 1000 else full_text
                snippet_parts.append(f"**Full text (excerpt):** {excerpt}")

            if result.get("methods"):
                snippet_parts.append(f"**Methods:** {result['methods']}")
            elif not result.get("has_fulltext"):
                if result.get("alternatives_found", 0) > 0:
                    snippet_parts.append(f"**Methods:** Abstract only in PubMed, but {result['alternatives_found']} alternative source(s) found with full text")
                else:
                    snippet_parts.append("**Methods:** Abstract only - no full text alternatives found")
            
            if result.get("url"):
                snippet_parts.append(f"**URL:** {result['url']}")
            
            if result.get("match_confidence"):
                snippet_parts.append(f"**Match Confidence:** {result['match_confidence']}")
            
            snippets.append("\n".join(snippet_parts))
            
        return snippets

    def _run(self, query: str, max_results: int = 20, include_preprints: bool = True) -> List[str]:
        """Main search function - PubMed first, then find alternatives for non-fulltext articles"""
        all_results = []
        
        # Step 1: Search PubMed/PMC first
        log.info("Searching PubMed/PMC...")
        pubmed_results = self.search_pubmed_pmc(query, max_results)
        
        if not pubmed_results:
            return ["No articles found in PubMed for your query."]
        
        # Step 2: Separate articles with and without full text
        fulltext_articles = []
        abstract_only_articles = []
        
        for article in pubmed_results:
            if article.get("has_fulltext", False):
                fulltext_articles.append(article)
            else:
                abstract_only_articles.append(article)
        
        log.info(f"Found {len(fulltext_articles)} articles with full text, {len(abstract_only_articles)} with abstract only")
        
        # Step 3: For abstract-only articles, search alternative databases
        alternative_results = []
        
        if abstract_only_articles and include_preprints:
            log.info(f"Searching alternative databases for {len(abstract_only_articles)} abstract-only articles...")
            
            for article in abstract_only_articles:
                log.info(f"Searching alternatives for: {article.get('title', '')[:50]}...")
                
                # Search for this specific article in alternative databases
                alternatives = self.search_alternative_sources(article, max_per_source=2)
                
                if alternatives:
                    # Mark the original article as having alternatives found
                    article["alternatives_found"] = len(alternatives)
                    alternative_results.extend(alternatives)
                    log.info(f"Found {len(alternatives)} alternative sources")
                else:
                    article["alternatives_found"] = 0
        
        # Step 4: Combine results
        all_results = fulltext_articles + abstract_only_articles + alternative_results
        
        # Step 5: Remove duplicates based on title similarity
        unique_results = []
        seen_titles = set()
        
        for result in all_results:
            title_key = result.get("title", "").lower().strip()[:50]
            if title_key and title_key not in seen_titles:
                seen_titles.add(title_key)
                unique_results.append(result)
        
        # Step 6: Sort results - full text first, then by source priority
        source_priority = {"PubMed": 1, "arXiv": 2, "bioRxiv": 2, "CrossRef": 3}
        unique_results.sort(key=lambda x: (
            not x.get("has_fulltext", False),  # Full text first
            source_priority.get(x.get("source", ""), 999),  # Then by source priority
            -x.get("relevance_score", 0)  # Then by relevance score
        ))
        
        # Step 7: Limit to requested number of results
        final_results = unique_results[:max_results]

        # Step 8: Format results
        formatted_snippets = self.format_results(final_results)

        # Step 9: Add comprehensive summary
        pubmed_count = sum(1 for r in final_results if r.get('source') == 'PubMed')
        arxiv_count = sum(1 for r in final_results if r.get('source') == 'arXiv')
        biorxiv_count = sum(1 for r in final_results if r.get('source') == 'bioRxiv')
        crossref_count = sum(1 for r in final_results if r.get('source') == 'CrossRef')
        fulltext_count = sum(1 for r in final_results if r.get('has_fulltext', False))
        abstract_only_count = len(final_results) - fulltext_count
        alternatives_found = sum(1 for r in final_results if r.get('alternatives_found', 0) > 0)

        summary = [
            f"SEARCH RESULTS SUMMARY:",
            f"Total articles found: {len(final_results)}",
            f"",
            f"By source:",
            f"  - PubMed/PMC: {pubmed_count} articles",
            f"  - arXiv: {arxiv_count} articles",
            f"  - bioRxiv: {biorxiv_count} articles",
            f"  - CrossRef: {crossref_count} articles",
            f"",
            f"Full-text availability:",
            f"  - Full text available: {fulltext_count} articles",
            f"  - Abstract only: {abstract_only_count} articles",
            f"  - Alternative sources found: {alternatives_found} articles",
            f"",
            f"STRATEGY: Searched PubMed first, then found alternative sources for abstract-only articles",
            "\n" + "="*80 + "\n",
        ]

        # Combine summary lines into a single string to avoid counting each line as a separate snippet
        summary_text = "\n".join(summary)

        # Return a single summary string followed by the formatted snippets
        return [summary_text] + formatted_snippets

In [8]:
# Quick smoke tests for the compound and journal tools
print("Running quick verification tests for CompoundDatabaseTool and FullTextJournalSearchTool...\n")

# Compound tool: name-based and smiles-based quick checks
try:
    print("1) CompoundDatabaseTool - name search ('aspirin')")
    out_name = compound_tool._run(compound_name="aspirin")
    print(str(out_name)[:3000])
except Exception as e:
    print("Compound name test error:", repr(e))

try:
    print("\n2) CompoundDatabaseTool - SMILES search (aspirin SMILES)")
    smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"
    out_smiles = compound_tool._run(smiles=smiles)
    print(str(out_smiles)[:3000])
except Exception as e:
    print("Compound SMILES test error:", repr(e))

# Journal tool: a short PubMed search (small result set)
try:
    print("\n3) FullTextJournalSearchTool - PubMed/PMC search ('single-cell RNA-seq methods')")
    snippets = journal_tool._run("single-cell RNA-seq methods", max_results=3, include_preprints=True)
    print(f"Returned {len(snippets)} snippet(s). Showing up to first 3 snippets (truncated).\n")
    for i, s in enumerate(snippets[:3], 1):
        print(f"--- Snippet {i} (truncated to 2000 chars) ---")
        try:
            print(s[:2000])
        except Exception:
            print(repr(s))
        print("\n")
except Exception as e:
    print("Journal tool test error:", repr(e))

print("\nSmoke tests completed.")

Running quick verification tests for CompoundDatabaseTool and FullTextJournalSearchTool...

1) CompoundDatabaseTool - name search ('aspirin')
## Compound Search Results

| Name | Formula | Mass (Da) | SMILES | PubChem CID |
|------|---------|-----------|--------|-------------|
| 2-acetyloxybenzoic acid | C9H8O4 | 180.159 | CC(=O)OC1=CC=CC=C1C(=O)O | [2244](https://pubchem.ncbi.nlm.nih.gov/compound/2244) |

### Primary Result Details:
**Name:** 2-acetyloxybenzoic acid
**Formula:** C9H8O4
**Molecular Weight:** 180.159 Da
**PubChem CID:** 2244
**SMILES:** CC(=O)OC1=CC=CC=C1C(=O)O

2) CompoundDatabaseTool - SMILES search (aspirin SMILES)


2025-10-01 17:33:23,477 | INFO | Searching PubMed/PMC...


## Compound Search Results

| Name | Formula | Mass (Da) | SMILES | PubChem CID |
|------|---------|-----------|--------|-------------|
| aspirin | C9H8O4 | 180.159 | CC(=O)OC1=CC=CC=C1C(=O)O | [2244](https://pubchem.ncbi.nlm.nih.gov/compound/2244) |

### Primary Result Details:
**Name:** aspirin
**Formula:** C9H8O4
**Molecular Weight:** 180.159 Da
**PubChem CID:** 2244
**SMILES:** CC(=O)OC1=CC=CC=C1C(=O)O
**Common Names:** aspirin, ACETYLSALICYLIC ACID, 50-78-2

### Molecular Analysis:
- Formula: C9H8O4
- Molecular Weight: 180.1590 Da
- Exact Mass: 180.0423 Da

3) FullTextJournalSearchTool - PubMed/PMC search ('single-cell RNA-seq methods')


2025-10-01 17:33:29,905 | INFO | Found 3 articles with full text, 0 with abstract only


Returned 19 snippet(s). Showing up to first 3 snippets (truncated).

--- Snippet 1 (truncated to 2000 chars) ---
SEARCH RESULTS SUMMARY:


--- Snippet 2 (truncated to 2000 chars) ---
Total articles found: 3


--- Snippet 3 (truncated to 2000 chars) ---




Smoke tests completed.
